# Dagalog Jupyter Kernel — Introduction

This notebook demonstrates the **Dagalog** kernel for interactive RDF data pipelines.
Each cell is a pipeline step; the in-memory `Datastore` persists across all cells in a session.

## Setup

```bash
# Build and install the kernel (run from repo root)
cargo build -p dagalog-kernel
./target/debug/dagalog-kernel install

# Launch (pins root_dir to the repo root so relative paths in %%rml/%%load work —
# Jupyter would otherwise default the kernel's working directory to notebooks/,
# since that's where this notebook file lives)
jupyter lab --ServerApp.root_dir=. notebooks/dagalog_intro.ipynb
```

Select **Dagalog (SPARQL + RDF)** as the kernel.

## Step 1 — Load inline Turtle

`%%turtle` parses the cell body as Turtle and adds the triples to the session datastore.

In [1]:
%%turtle
@prefix ex:   <http://example.com/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:Alice a foaf:Person ; foaf:name "Alice" ; foaf:age 30 ; ex:worksFor ex:Acme .
ex:Bob   a foaf:Person ; foaf:name "Bob"   ; foaf:age 25 ; ex:worksFor ex:Acme .
ex:Acme  a ex:Company  ; foaf:name "Acme Corp" .

Loaded 10 triples.

## Step 2 — Query with SPARQL SELECT

Cells without a `%%` prefix are treated as SPARQL. SELECT results render as an HTML table.

In [2]:
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.com/>

SELECT ?person ?name ?age WHERE {
    ?person a foaf:Person ;
            foaf:name ?name ;
            foaf:age  ?age .
}
ORDER BY ?name

person,name,age
<http://example.com/Alice>,(Alice),30^^http://www.w3.org/2001/XMLSchema#integer
<http://example.com/Bob>,(Bob),25^^http://www.w3.org/2001/XMLSchema#integer


## Step 3 — Apply an RML CSV mapping

`%%rml` loads an [RML mapping file](https://rml.io/) and runs it against
the declared CSV source, adding the resulting triples to the session datastore.

Paths are relative to the kernel's working directory, which is `--ServerApp.root_dir`
(the repo root, per the setup instructions above) — **not** the directory the
notebook file lives in.

In [3]:
%%rml ../tests/testdata/rml_persons_mapping.ttl

Loaded 6 triples.

## Step 4 — OWL-RL Reasoning

`%%reason` runs OWL-RL materialisation on the current datastore, inferring new triples.

In [4]:
%%reason

Reasoning complete. 0 triples added.

## Step 5 — Inspect the graph after loading

Query to see all people known to the datastore.

In [5]:
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
PREFIX ex:   <http://example.com/>

SELECT ?person ?name WHERE {
    ?person a foaf:Person ;
            foaf:name ?name .
}
ORDER BY ?name

person,name
<http://example.com/Alice>,(Alice)
<http://example.com/Bob>,(Bob)


## Step 6 — Inline Datalog rules

`%%datalog` parses and materialises forward-chaining Datalog rules.

In [6]:
%%datalog
[?x, <http://example.com/colleague>, ?y] :-
    [?x, <http://example.com/worksFor>, ?org],
    [?y, <http://example.com/worksFor>, ?org] .

Applied 1 rule.

In [7]:
PREFIX ex: <http://example.com/>

SELECT ?person ?colleague WHERE {
    ?person ex:colleague ?colleague .
    FILTER (?person != ?colleague)
}
ORDER BY ?person ?colleague

person,colleague
<http://example.com/Alice>,<http://example.com/Bob>
<http://example.com/Bob>,<http://example.com/Alice>
